# Lead CRM – PostgreSQL Relational Database using SQLAlchemy

This follows the structure of the supplied `MainProject-SqlFinal` sample:
connect → import tables → validate keys → load tables → primary keys → foreign keys → date dimension → verification.

## 1. Connect to PostgreSQL

In [4]:
import pandas as pd
from sqlalchemy import create_engine, text

username = "postgres"
password = "ramshishabi"
host = "localhost"
port = "5432"
database = "crm_leads_main_project"

engine = create_engine(
    f"postgresql+psycopg2://{username}:{password}@{host}:{port}/{database}"
)

with engine.connect() as conn:
    conn.execute(text("SELECT 1"))

print("Connected successfully!")

Connected successfully!


## 2. Import the relational tables

In [5]:
TABLE_DIR = "database-data"

dim_user = pd.read_csv(f"{TABLE_DIR}/dim_user.csv")
dim_source = pd.read_csv(f"{TABLE_DIR}/dim_source.csv")
dim_campaign = pd.read_csv(f"{TABLE_DIR}/dim_campaign.csv")
dim_contact = pd.read_csv(f"{TABLE_DIR}/dim_contact.csv")
fact_lead = pd.read_csv(f"{TABLE_DIR}/fact_lead.csv")
fact_lead_activity = pd.read_csv(f"{TABLE_DIR}/fact_lead_activity.csv")

datetime_by_table = {
    "fact_lead": [
        "lead_date", "won_date", "expected_closing_date",
        "lead_assigned_at", "first_touch_at", "tomorrow_closing",
        "created_time", "updated_time", "viewed_time"
    ],
    "fact_lead_activity": ["last_call_time", "next_follow_up_on"]
}

for col in datetime_by_table["fact_lead"]:
    if col in fact_lead.columns:
        fact_lead[col] = pd.to_datetime(fact_lead[col], errors="coerce")

for col in datetime_by_table["fact_lead_activity"]:
    if col in fact_lead_activity.columns:
        fact_lead_activity[col] = pd.to_datetime(fact_lead_activity[col], errors="coerce")

tables = {
    "dim_user": dim_user,
    "dim_source": dim_source,
    "dim_campaign": dim_campaign,
    "dim_contact": dim_contact,
    "fact_lead": fact_lead,
    "fact_lead_activity": fact_lead_activity
}

for name, table in tables.items():
    print(name, table.shape)

dim_user (15, 2)
dim_source (18, 2)
dim_campaign (158, 7)
dim_contact (10000, 10)
fact_lead (10000, 27)
fact_lead_activity (10000, 10)


## 3. Check primary-key columns

In [6]:
key_checks = {
    "dim_user.user_id": dim_user["user_id"],
    "dim_source.source_id": dim_source["source_id"],
    "dim_campaign.campaign_id": dim_campaign["campaign_id"],
    "dim_contact.contact_id": dim_contact["contact_id"],
    "fact_lead.lead_id": fact_lead["lead_id"],
    "fact_lead_activity.activity_id": fact_lead_activity["activity_id"]
}

for name, series in key_checks.items():
    print(
        f"{name}: rows={len(series)}, "
        f"unique={series.nunique(dropna=False)}, "
        f"nulls={series.isna().sum()}"
    )

dim_user.user_id: rows=15, unique=15, nulls=0
dim_source.source_id: rows=18, unique=18, nulls=0
dim_campaign.campaign_id: rows=158, unique=158, nulls=0
dim_contact.contact_id: rows=10000, unique=10000, nulls=0
fact_lead.lead_id: rows=10000, unique=10000, nulls=0
fact_lead_activity.activity_id: rows=10000, unique=10000, nulls=0


## 4. Remove old tables so the notebook is rerunnable

In [7]:
with engine.begin() as conn:
    conn.execute(text("DROP TABLE IF EXISTS fact_lead_activity CASCADE"))
    conn.execute(text("DROP TABLE IF EXISTS fact_lead CASCADE"))
    conn.execute(text("DROP TABLE IF EXISTS dim_date CASCADE"))
    conn.execute(text("DROP TABLE IF EXISTS dim_campaign CASCADE"))
    conn.execute(text("DROP TABLE IF EXISTS dim_contact CASCADE"))
    conn.execute(text("DROP TABLE IF EXISTS dim_source CASCADE"))
    conn.execute(text("DROP TABLE IF EXISTS dim_user CASCADE"))

print("Old Lead CRM tables removed successfully!")

Old Lead CRM tables removed successfully!


## 5. Load tables using SQLAlchemy

In [8]:
dim_user.to_sql("dim_user", engine, if_exists="replace", index=False)
dim_source.to_sql("dim_source", engine, if_exists="replace", index=False)
dim_campaign.to_sql("dim_campaign", engine, if_exists="replace", index=False)
dim_contact.to_sql("dim_contact", engine, if_exists="replace", index=False)
fact_lead.to_sql("fact_lead", engine, if_exists="replace", index=False)
fact_lead_activity.to_sql("fact_lead_activity", engine, if_exists="replace", index=False)

print("All relational tables loaded successfully!")

All relational tables loaded successfully!


## 6. Add primary keys

In [9]:
with engine.begin() as conn:
    conn.execute(text(
        "ALTER TABLE dim_user ADD CONSTRAINT pk_dim_user PRIMARY KEY (user_id)"
    ))
    conn.execute(text(
        "ALTER TABLE dim_source ADD CONSTRAINT pk_dim_source PRIMARY KEY (source_id)"
    ))
    conn.execute(text(
        "ALTER TABLE dim_campaign ADD CONSTRAINT pk_dim_campaign PRIMARY KEY (campaign_id)"
    ))
    conn.execute(text(
        "ALTER TABLE dim_contact ADD CONSTRAINT pk_dim_contact PRIMARY KEY (contact_id)"
    ))
    conn.execute(text(
        "ALTER TABLE fact_lead ADD CONSTRAINT pk_fact_lead PRIMARY KEY (lead_id)"
    ))
    conn.execute(text(
        "ALTER TABLE fact_lead_activity ADD CONSTRAINT pk_fact_lead_activity PRIMARY KEY (activity_id)"
    ))

print("Primary keys created successfully!")

Primary keys created successfully!


## 7. Check foreign-key values before creating relationships

In [10]:
checks = {
    "lead -> user": '''
        SELECT COUNT(*) AS missing
        FROM fact_lead f
        LEFT JOIN dim_user u ON f.user_id = u.user_id
        WHERE f.user_id IS NOT NULL AND u.user_id IS NULL
    ''',
    "lead -> source": '''
        SELECT COUNT(*) AS missing
        FROM fact_lead f
        LEFT JOIN dim_source s ON f.source_id = s.source_id
        WHERE f.source_id IS NOT NULL AND s.source_id IS NULL
    ''',
    "lead -> campaign": '''
        SELECT COUNT(*) AS missing
        FROM fact_lead f
        LEFT JOIN dim_campaign c ON f.campaign_id = c.campaign_id
        WHERE f.campaign_id IS NOT NULL AND c.campaign_id IS NULL
    ''',
    "lead -> contact": '''
        SELECT COUNT(*) AS missing
        FROM fact_lead f
        LEFT JOIN dim_contact c ON f.contact_id = c.contact_id
        WHERE f.contact_id IS NOT NULL AND c.contact_id IS NULL
    ''',
    "activity -> lead": '''
        SELECT COUNT(*) AS missing
        FROM fact_lead_activity a
        LEFT JOIN fact_lead f ON a.lead_id = f.lead_id
        WHERE f.lead_id IS NULL
    '''
}

for name, query in checks.items():
    result = pd.read_sql(query, engine)
    print(f"{name}: {int(result.iloc[0]['missing'])} missing")

lead -> user: 0 missing
lead -> source: 0 missing
lead -> campaign: 0 missing
lead -> contact: 0 missing
activity -> lead: 0 missing


## 8. Add foreign keys

In [11]:
with engine.begin() as conn:
    conn.execute(text(
        "ALTER TABLE fact_lead "
        "ADD CONSTRAINT fk_fact_lead_user "
        "FOREIGN KEY (user_id) REFERENCES dim_user(user_id)"
    ))
    conn.execute(text(
        "ALTER TABLE fact_lead "
        "ADD CONSTRAINT fk_fact_lead_source "
        "FOREIGN KEY (source_id) REFERENCES dim_source(source_id)"
    ))
    conn.execute(text(
        "ALTER TABLE fact_lead "
        "ADD CONSTRAINT fk_fact_lead_campaign "
        "FOREIGN KEY (campaign_id) REFERENCES dim_campaign(campaign_id)"
    ))
    conn.execute(text(
        "ALTER TABLE fact_lead "
        "ADD CONSTRAINT fk_fact_lead_contact "
        "FOREIGN KEY (contact_id) REFERENCES dim_contact(contact_id)"
    ))
    conn.execute(text(
        "ALTER TABLE fact_lead_activity "
        "ADD CONSTRAINT fk_activity_lead "
        "FOREIGN KEY (lead_id) REFERENCES fact_lead(lead_id)"
    ))

print("Foreign key relationships created successfully!")

Foreign key relationships created successfully!


## 9. Create the date dimension

In [12]:
with engine.begin() as conn:
    conn.execute(text('''
        CREATE TABLE dim_date AS
        SELECT DISTINCT lead_date::date AS date
        FROM fact_lead
        WHERE lead_date IS NOT NULL
    '''))

    conn.execute(text('''
        ALTER TABLE dim_date
        ADD COLUMN year INTEGER,
        ADD COLUMN quarter INTEGER,
        ADD COLUMN month INTEGER,
        ADD COLUMN month_name VARCHAR(20),
        ADD COLUMN week_number INTEGER,
        ADD COLUMN day INTEGER,
        ADD COLUMN day_name VARCHAR(20)
    '''))

    conn.execute(text('''
        UPDATE dim_date
        SET
            year = EXTRACT(YEAR FROM date)::INTEGER,
            quarter = EXTRACT(QUARTER FROM date)::INTEGER,
            month = EXTRACT(MONTH FROM date)::INTEGER,
            month_name = TRIM(TO_CHAR(date, 'Month')),
            week_number = EXTRACT(WEEK FROM date)::INTEGER,
            day = EXTRACT(DAY FROM date)::INTEGER,
            day_name = TRIM(TO_CHAR(date, 'Day'))
    '''))

    conn.execute(text(
        "ALTER TABLE dim_date "
        "ADD CONSTRAINT pk_dim_date PRIMARY KEY (date)"
    ))

print("dim_date created successfully!")

dim_date created successfully!


## 10. Add lead-date relationship

In [13]:
with engine.begin() as conn:
    conn.execute(text(
        "ALTER TABLE fact_lead ADD COLUMN lead_date_key DATE"
    ))
    conn.execute(text('''
        UPDATE fact_lead
        SET lead_date_key = lead_date::date
        WHERE lead_date IS NOT NULL
    '''))
    conn.execute(text(
        "ALTER TABLE fact_lead "
        "ADD CONSTRAINT fk_fact_lead_date "
        "FOREIGN KEY (lead_date_key) REFERENCES dim_date(date)"
    ))

print("Lead-to-date relationship created successfully!")

Lead-to-date relationship created successfully!


## 11. Verify tables

In [14]:
tables_in_db = pd.read_sql('''
    SELECT table_name
    FROM information_schema.tables
    WHERE table_schema = 'public'
    ORDER BY table_name
''', engine)

display(tables_in_db)

,table_name
0,dim_campaign
1,dim_contact
2,dim_date
3,dim_source
4,dim_user
5,fact_lead
6,fact_lead_activity


## 12. Verify primary and foreign keys

In [15]:
constraints = pd.read_sql('''
SELECT
    tc.constraint_name,
    tc.constraint_type,
    tc.table_name,
    kcu.column_name,
    ccu.table_name AS referenced_table,
    ccu.column_name AS referenced_column
FROM information_schema.table_constraints tc
LEFT JOIN information_schema.key_column_usage kcu
    ON tc.constraint_name = kcu.constraint_name
    AND tc.table_schema = kcu.table_schema
LEFT JOIN information_schema.constraint_column_usage ccu
    ON ccu.constraint_name = tc.constraint_name
    AND ccu.table_schema = tc.table_schema
WHERE tc.table_schema = 'public'
ORDER BY tc.table_name, tc.constraint_type, tc.constraint_name
''', engine)

display(constraints)

,constraint_name,constraint_type,table_name,column_name,referenced_table,referenced_column
0,dim_campaign_campaign_id_not_null,CHECK,dim_campaign,None,dim_campaign,campaign_id
1,pk_dim_campaign,PRIMARY KEY,dim_campaign,campaign_id,dim_campaign,campaign_id
2,dim_contact_contact_id_not_null,CHECK,dim_contact,None,dim_contact,contact_id
3,pk_dim_contact,PRIMARY KEY,dim_contact,contact_id,dim_contact,contact_id
4,dim_date_date_not_null,CHECK,dim_date,None,dim_date,date
5,pk_dim_date,PRIMARY KEY,dim_date,date,dim_date,date
6,dim_source_source_id_not_null,CHECK,dim_source,None,dim_source,source_id
7,pk_dim_source,PRIMARY KEY,dim_source,source_id,dim_source,source_id
8,dim_user_user_id_not_null,CHECK,dim_user,None,dim_user,user_id
9,pk_dim_user,PRIMARY KEY,dim_user,user_id,dim_user,user_id


## 13. Final row-count check

In [16]:
row_counts = pd.read_sql('''
SELECT 'dim_user' AS table_name, COUNT(*) AS row_count FROM dim_user
UNION ALL
SELECT 'dim_source', COUNT(*) FROM dim_source
UNION ALL
SELECT 'dim_campaign', COUNT(*) FROM dim_campaign
UNION ALL
SELECT 'dim_contact', COUNT(*) FROM dim_contact
UNION ALL
SELECT 'fact_lead', COUNT(*) FROM fact_lead
UNION ALL
SELECT 'fact_lead_activity', COUNT(*) FROM fact_lead_activity
UNION ALL
SELECT 'dim_date', COUNT(*) FROM dim_date
ORDER BY table_name
''', engine)

display(row_counts)
print("\nLead CRM relational database setup completed successfully!")

,table_name,row_count
0,dim_campaign,158
1,dim_contact,10000
2,dim_date,48
3,dim_source,18
4,dim_user,15
5,fact_lead,10000
6,fact_lead_activity,10000



Lead CRM relational database setup completed successfully!
